# Gemma 4 31B Tool-Calling Fine-Tune Cloud Stable

This cloud-stable variant keeps the same 31B / 4-bit / 4K-context quality target, but uses safer Unsloth-aligned settings to reduce notebook kernel crash risk.

**Changes**
- LoRA rank reduced from 32 to 16
- Unsloth gradient checkpointing enabled
- bounded `max_steps` chunk instead of a long 2-epoch run
- checkpoints every 10 steps
- first pass uses a smaller dataset slice


### Installation


In [ ]:
# === Create a clean AMD/Unsloth kernel (doc-aligned path) ===
# This follows the official AMD + Unsloth guidance: isolated env, ROCm PyTorch, then Unsloth.
import os, re, shutil, subprocess, sys
from pathlib import Path
INSTALL_ROOT = os.environ.get('UNSLOTH_INSTALL_ROOT', str(Path.home() / 'unsloth-gemma4'))
VENV = os.path.join(INSTALL_ROOT, 'venv')
TMPDIR = os.path.join(INSTALL_ROOT, 'tmp')
PIP_CACHE_DIR = os.path.join(INSTALL_ROOT, 'pip-cache')
PY = os.path.join(VENV, 'bin', 'python')
PIP = [PY, '-m', 'pip']

def run(cmd):
    print('\n$ ' + ' '.join(cmd))
    env = os.environ.copy()
    env['TMPDIR'] = TMPDIR
    env['TEMP'] = TMPDIR
    env['TMP'] = TMPDIR
    env['PIP_CACHE_DIR'] = PIP_CACHE_DIR
    r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if r.stdout.strip():
        out = r.stdout.strip().splitlines()
        for line in out[-8:]:
            print(line)
    if r.returncode != 0:
        err = r.stderr.strip() or '(no stderr)'
        raise RuntimeError(err)

def detect_rocm_tag():
    candidates = []
    try:
        r = subprocess.run(['amd-smi', 'version'], capture_output=True, text=True)
        candidates.append(r.stdout)
    except FileNotFoundError:
        pass
    try:
        candidates.append(Path('/opt/rocm/.info/version').read_text())
    except Exception:
        pass
    try:
        r = subprocess.run(['hipconfig', '--version'], capture_output=True, text=True)
        candidates.append(r.stdout)
    except FileNotFoundError:
        pass
    for text in candidates:
        m = re.search(r'(?:ROCm version: |HIP version: )([0-9]+)\.([0-9]+)', text)
        if not m:
            m = re.search(r'([0-9]+)\.([0-9]+)', text)
        if m:
            return f'rocm{m.group(1)}.{m.group(2)}'
    return 'rocm7.0'

Path(INSTALL_ROOT).mkdir(parents=True, exist_ok=True)
Path(TMPDIR).mkdir(parents=True, exist_ok=True)
Path(PIP_CACHE_DIR).mkdir(parents=True, exist_ok=True)
rocm_tag = detect_rocm_tag()
print('ROCm wheel tag:', rocm_tag)
print('Install root:', INSTALL_ROOT)
print('Temp dir:', TMPDIR)
print('Pip cache:', PIP_CACHE_DIR)

if os.path.isdir(VENV):
    shutil.rmtree(VENV)
run([sys.executable, '-m', 'venv', VENV])
run(PIP + ['install', '--upgrade', 'pip', 'setuptools', 'wheel', 'ipykernel'])
run(PIP + ['install', '--upgrade', '--force-reinstall', 'torch', 'torchvision', 'torchaudio', '--index-url', f'https://download.pytorch.org/whl/{rocm_tag}'])
run(PIP + ['install', '--no-deps', 'unsloth', 'unsloth-zoo'])
run(PIP + ['install', '--no-deps', 'git+https://github.com/unslothai/unsloth-zoo.git'])
run(PIP + ['install', 'unsloth[amd] @ git+https://github.com/unslothai/unsloth'])
run(PIP + ['install', '--upgrade', 'git+https://github.com/huggingface/transformers.git'])
run(PIP + ['install', '--upgrade', 'timm', 'torchcodec'])
run([PY, '-m', 'ipykernel', 'install', '--user', '--name', 'unsloth-gemma4-amd', '--display-name', 'Python (unsloth-gemma4-amd)'])

print('\nKernel created: Python (unsloth-gemma4-amd)')
print('Next: Kernel -> Change Kernel -> Python (unsloth-gemma4-amd)')
print('Then restart kernel and run Cell 3.')


In [ ]:
# === Verify you are on the clean Unsloth AMD kernel ===
import os, sys
from pathlib import Path
VENV = os.environ.get('UNSLOTH_VENV', str(Path.home() / 'unsloth-gemma4' / 'venv'))
print('sys.executable:', sys.executable)
print('sys.prefix:', sys.prefix)
print('VIRTUAL_ENV:', os.environ.get('VIRTUAL_ENV'))
on_expected_kernel = (
    sys.prefix == VENV
    or sys.executable.startswith(VENV + os.sep)
    or os.environ.get('VIRTUAL_ENV') == VENV
)
if not on_expected_kernel:
    raise RuntimeError(
        'Wrong kernel. Change kernel to Python (unsloth-gemma4-amd), then restart and re-run this cell.'
    )

import unsloth, unsloth_zoo, huggingface_hub, regex, transformers
from transformers import AutoConfig

print(f'unsloth: {getattr(unsloth, "__version__", "OK")} ({unsloth.__file__})')
print(f'unsloth_zoo: {getattr(unsloth_zoo, "__version__", "OK")} ({unsloth_zoo.__file__})')
print(f'huggingface_hub: {huggingface_hub.__version__} ({huggingface_hub.__file__})')
print(f'regex: {regex.__version__} ({regex.__file__})')
print(f'transformers: {transformers.__version__} ({transformers.__file__})')
c = AutoConfig.from_pretrained("unsloth/gemma-4-26B-A4B-it")
print(f'AutoConfig OK: model_type={c.model_type}')

import torch; torch._dynamo.config.recompile_limit = 64
print('Next: run the HF token cell, then the Load Model cell.')


In [ ]:
import os
from pathlib import Path

# === HuggingFace Configuration ===
# Option 1: Set env var before launching notebook: export HF_TOKEN=hf_...
# Option 2: Uncomment and paste below:
# os.environ["HF_TOKEN"] = "hf_your_actual_token"

HF_TOKEN = os.environ.get("HF_TOKEN", "")
HF_REPO_ID = "mtita/gemma4-31b-tool-calling-lora"

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN":
    print("\u26a0\ufe0f  No HF token set. Hub pushing disabled.")
    print("   To enable: set HF_TOKEN env var before running.")
    HF_TOKEN = ""
    PUSH_TO_HUB = False
else:
    PUSH_TO_HUB = True

# === Training Output ===
TRAINING_OUTPUT_DIR = os.environ.get(
    "TRAINING_OUTPUT_DIR",
    str(Path.home() / "gemma4_runs" / "gemma4_toolcalling_sft")
)
Path(TRAINING_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("HF repo:", HF_REPO_ID)
print("Push enabled:", PUSH_TO_HUB)
print("Training outputs:", TRAINING_OUTPUT_DIR)

### Load Model


In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-26B-A4B-it",
    dtype = torch.bfloat16,
    max_seq_length = 4096,
    load_in_4bit = False,
    full_finetuning = False,
    token = HF_TOKEN or None,
)

### Add LoRA Adapters


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Text-only
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_gradient_checkpointing = "unsloth",
)


### Data Prep

We use the `gemma-4` chat template. Gemma 4 renders conversations like:
```
<bos><|turn>user
Hello<turn|>
<|turn>model
Hey there!<turn|>
```


In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)


In [ ]:
from datasets import load_dataset

# Start with a smaller slice to prove cloud stability, then widen later.
DATASET_SPLIT = "train[:3000]"
dataset = load_dataset(
    "mtita/gemma4-tool-calling-training",
    data_files="combined_training.jsonl",
    split=DATASET_SPLIT,
)
print(f"Loaded {len(dataset)} training examples from {DATASET_SPLIT}")
print(dataset[0]["messages"][:2])


Map roles for Gemma 4 (system → user prefix, tool → user prefix) and apply chat template.


In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        formatted = []
        for msg in messages:
            role = msg["role"]
            content = msg.get("content", "") or ""
            if role == "system":
                sys_msg = {"role": "user", "content": f"[System Instructions]\n{content}"}
                ack_msg = {"role": "assistant", "content": "Understood."}
                # Merge with previous user if consecutive
                if formatted and formatted[-1]["role"] == "user":
                    formatted[-1]["content"] += f"\n\n{sys_msg['content']}"
                    formatted.append(ack_msg)
                else:
                    formatted.append(sys_msg)
                    formatted.append(ack_msg)
            elif role == "user":
                if formatted and formatted[-1]["role"] == "user":
                    formatted[-1]["content"] += f"\n\n{content}"
                else:
                    formatted.append({"role": "user", "content": content})
            elif role == "assistant":
                if formatted and formatted[-1]["role"] == "assistant":
                    formatted[-1]["content"] += f"\n\n{content}"
                else:
                    formatted.append({"role": "assistant", "content": content})
            elif role == "tool":
                tool_name = msg.get("name", "tool")
                tool_text = f"[Tool Result: {tool_name}]\n{content}"
                if formatted and formatted[-1]["role"] == "user":
                    formatted[-1]["content"] += f"\n\n{tool_text}"
                else:
                    formatted.append({"role": "user", "content": tool_text})
        # Ensure starts with user
        if not formatted or formatted[0]["role"] != "user":
            texts.append("")
            continue
        # Ensure ends with assistant (we want to train on the final response)
        if formatted[-1]["role"] != "assistant":
            texts.append("")
            continue
        try:
            text = tokenizer.apply_chat_template(
                formatted, tokenize=False, add_generation_prompt=False
            ).removeprefix('<bos>')
            texts.append(text)
        except Exception:
            texts.append("")
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
dataset = dataset.filter(lambda x: len(x["text"]) > 50)
print(f"Formatted: {len(dataset)} examples")
print(dataset[0]["text"][:500])

### Stability strategy

Training runs for **2 full epochs** (~15k steps). Checkpoints are saved every 100 steps. If the kernel dies, re-run the training cell — it will auto-resume from the last checkpoint.

If this stays stable, scale in this order:
1. increase dataset slice
2. adjust `num_train_epochs` or `save_steps`
3. increase LoRA rank

### Train the model


In [ ]:
from transformers.trainer_utils import get_last_checkpoint

LAST_CHECKPOINT = get_last_checkpoint(TRAINING_OUTPUT_DIR)
print("Latest checkpoint:", LAST_CHECKPOINT or "none")


Only train on assistant outputs, ignore user inputs:


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 4096,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        num_train_epochs = 2,
        warmup_steps = 5,
        learning_rate = 1e-4,
        logging_steps = 1,
        optim = "adamw_torch",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        output_dir = TRAINING_OUTPUT_DIR,
        save_strategy = "no",
        # save_steps = 50,
        # save_total_limit = 5,
        hub_model_id = HF_REPO_ID,
        push_to_hub = PUSH_TO_HUB,
        hub_strategy = "checkpoint",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)


### Verify masking before training

Run these to confirm only assistant responses are being trained on.

In [ ]:
# Verify masking - should see full conversation
tokenizer.decode(trainer.train_dataset[0]["input_ids"])


In [ ]:
# Verify masking - should see ONLY assistant responses (rest is spaces)
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")


### Start or resume training

If the kernel dies or the notebook disconnects, re-open the notebook, run the setup cells again, then run the cell below.


In [ ]:
train_result = trainer.train(resume_from_checkpoint=LAST_CHECKPOINT)
train_result


### Push final adapter to Hugging Face

Use this after training completes so you can always restore from Hub even if the cloud VM is wiped.


In [ ]:
if PUSH_TO_HUB:
    trainer.push_to_hub()
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print("Pushed final training artifacts to", HF_REPO_ID)
else:
    print("Skipping push_to_hub")


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


### Test Inference


In [ ]:
# Quick inference test
messages = [{"role": "user",
"content": [{"type": "text", "text": "You have these tools:\n"
"- read(path): Read a file\n"
"- edit(path, old_text, new_text): Replace text in a file\n"
"- bash(command): Run a shell command\n\n"
"Fix the typo in src/utils.py where 'retrun' should be 'return'."}]}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 512,
                    temperature = 0.7, top_p = 0.9)

### Save LoRA adapters

**Set your HF token below!**


In [ ]:
model.save_pretrained("gemma_4_lora")
tokenizer.save_pretrained("gemma_4_lora")

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print("Pushed LoRA adapter to", HF_REPO_ID)
else:
    print("Saved locally. Set HF_TOKEN to push to Hub.")

### Export to GGUF

Supported: q4_k_m (recommended), q5_k_m, q8_0, f16, bf16 and more.


In [ ]:
model.save_pretrained_gguf(
    "gemma_4_finetune",
    tokenizer,
    quantization_method = "q4_k_m",
)


In [ ]:
# Push GGUF to HuggingFace
if PUSH_TO_HUB:
    model.push_to_hub_gguf(
        "mtita/gemma4-31b-tool-calling-q4_k_m-GGUF",
        tokenizer,
        quantization_method = "q4_k_m",
        token = HF_TOKEN,
    )
    print("Pushed GGUF to Hub")
else:
    print("Saved GGUF locally. Set HF_TOKEN to push.")